# DIP-style unsupervised dehazing
Inspired by Deep Image Prior (Ulyanov et al., CVPR 2018) and Double-DIP (Gandelsman et al., CVPR 2019). Each image is optimized from scratch: no training data, no pretrained weights. ~5 min per image on GPU.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# DIP-style unsupervised dehazing (inspired by Deep Image Prior, Ulyanov et al. CVPR 2018, and Double-DIP,
# Gandelsman et al. CVPR 2019). Each image is optimized from scratch: no training data, no pretrained weights.
print(torch.__version__, "| CUDA:", torch.cuda.is_available())

## Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE     = "/content/drive/MyDrive"
SRC       = f"{DRIVE}/O-HAZE"          # original O-HAZE: hazy/ + GT/
DATA_ROOT = f"{DRIVE}/O-HAZE-1600"     # resized copy, shared by all methods and the evaluation
OUT       = f"{DRIVE}/outputs/dip" # dehazed images, same file names as hazy/

MAX_IMAGES = 5             # first N pairs (None = all 45)
RESIZE = 1600              # long side; O-HAZE is ~3000x4000

In [ ]:
#resize hazy + GT once (skipped if DATA_ROOT already exists), same resizing as before:
#hazy: long side -> RESIZE (bicubic), GT: resized to the hazy size
import os, glob
from PIL import Image

def resize_long(img, long_side):
    if long_side <= 0 or max(img.size) <= long_side:
        return img
    s = long_side / max(img.size)
    return img.resize((round(img.size[0] * s), round(img.size[1] * s)), Image.BICUBIC)

if not os.path.isdir(f"{DATA_ROOT}/hazy"):
    os.makedirs(f"{DATA_ROOT}/hazy"); os.makedirs(f"{DATA_ROOT}/GT")
    pairs = list(zip(sorted(glob.glob(f"{SRC}/hazy/*")), sorted(glob.glob(f"{SRC}/GT/*"))))[:MAX_IMAGES]
    for h, g in pairs:
        hazy = resize_long(Image.open(h).convert('RGB'), RESIZE)
        gt   = Image.open(g).convert('RGB').resize(hazy.size, Image.BICUBIC)
        stem = lambda p: os.path.splitext(os.path.basename(p))[0]
        hazy.save(f"{DATA_ROOT}/hazy/{stem(h)}.png"); gt.save(f"{DATA_ROOT}/GT/{stem(g)}.png")

hazy_paths = sorted(glob.glob(f"{DATA_ROOT}/hazy/*"))
os.makedirs(OUT, exist_ok=True)
print(len(hazy_paths), "images:", [os.path.basename(p) for p in hazy_paths])

## Parameters

In [ ]:
# optimization resolution: t and A are optimized on a downscaled copy, then t is upsampled to RESIZE
OPT_SIZE = 512

# DIP networks
NOISE_CH = 32              # channels of the fixed random input z (DIP default)
NET_CH = 64                # feature channels of the encoder-decoder
INPUT_NOISE_STD = 1 / 30   # small perturbation added to z every iteration (DIP regularization trick)

# optimization -- fixed schedule, no early stopping on GT
NUM_ITER = 2000
LR = 1e-3                  # Adam
SEED = 0                   # random init of z and the networks -> fix it for reproducibility
LOG_EVERY = 500

# loss = MSE(J*t + A*(1-t), I) + LAMBDA_DC * mean(dark_channel(J))
LAMBDA_DC = 0.01           # weight of the dark channel loss; 0 -> trivial solution (t = 1, J = I)
DC_PATCH = 15              # patch size for the dark channel loss (matches OPT_SIZE scale)
T_MIN = 0.1                # t is mapped to [T_MIN, 1], same role as T0 in DCP

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

## Network

In [ ]:
def conv_block(i, o, k=3, stride=1):
    return nn.Sequential(nn.Conv2d(i, o, k, stride, k // 2, padding_mode='reflect'),
                         nn.BatchNorm2d(o), nn.LeakyReLU(0.2, inplace=True))

class DIPNet(nn.Module):
    """Encoder-decoder with skip connections (simplified 'skip' net from Deep Image Prior)."""
    def __init__(self, in_ch, out_ch, ch=NET_CH, depth=4, skip_ch=4):
        super().__init__()
        self.skips = nn.ModuleList([conv_block(in_ch if d == 0 else ch, skip_ch, k=1) for d in range(depth)])
        self.downs = nn.ModuleList([nn.Sequential(conv_block(in_ch if d == 0 else ch, ch, stride=2),
                                                  conv_block(ch, ch)) for d in range(depth)])
        self.ups   = nn.ModuleList([nn.Sequential(conv_block(ch + skip_ch, ch), conv_block(ch, ch, k=1))
                                    for _ in range(depth)])
        self.out = nn.Conv2d(ch, out_ch, 1)

    def forward(self, x):
        skips = []
        for skip, down in zip(self.skips, self.downs):
            skips.append(skip(x))
            x = down(x)
        for up, s in zip(self.ups, reversed(skips)):
            x = F.interpolate(x, size=s.shape[-2:], mode='bilinear', align_corners=False)   # handles odd sizes
            x = up(torch.cat([x, s], dim=1))
        return self.out(x)


def dark_channel_t(x, patch=DC_PATCH):
    """Differentiable dark channel: min over RGB, then min-pool over a patch (= -maxpool(-x))."""
    return -F.max_pool2d(-x.min(dim=1, keepdim=True)[0], patch, stride=1, padding=patch // 2)


to_tensor = lambda img: torch.from_numpy(np.asarray(img, dtype=np.float32) / 255).permute(2, 0, 1)[None].to(DEVICE)


def dehaze_dip(hazy):
    """hazy: PIL image at evaluation resolution. Returns (J, t, A, loss history) at that resolution."""
    torch.manual_seed(SEED)                                   # same init for every image -> reproducible

    I_low = to_tensor(resize_long(hazy, OPT_SIZE))           # optimization happens at OPT_SIZE
    _, _, h, w = I_low.shape

    z_J = torch.rand(1, NOISE_CH, h, w, device=DEVICE) * 0.1  # fixed random inputs (DIP)
    z_t = torch.rand(1, NOISE_CH, h, w, device=DEVICE) * 0.1
    J_net = DIPNet(NOISE_CH, 3).to(DEVICE)
    t_net = DIPNet(NOISE_CH, 1).to(DEVICE)

    # A: learnable RGB constant, initialised from the brightest 0.1% pixels of the input (no dark channel involved)
    flat = I_low[0].reshape(3, -1)
    top = flat.sum(0).topk(max(int(flat.shape[1] * 0.001), 1)).indices
    A0 = flat[:, top].mean(1).clamp(0.05, 0.99)
    a_logit = torch.logit(A0).detach().clone().requires_grad_(True)

    opt = torch.optim.Adam(list(J_net.parameters()) + list(t_net.parameters()) + [a_logit], lr=LR)
    hist = []
    for it in range(1, NUM_ITER + 1):
        J = torch.sigmoid(J_net(z_J + INPUT_NOISE_STD * torch.randn_like(z_J)))
        t = T_MIN + (1 - T_MIN) * torch.sigmoid(t_net(z_t + INPUT_NOISE_STD * torch.randn_like(z_t)))
        A = torch.sigmoid(a_logit).view(1, 3, 1, 1)

        loss_rec = F.mse_loss(J * t + A * (1 - t), I_low)    # must reproduce the hazy input
        loss_dc  = dark_channel_t(J).mean()                   # pushes J towards haze-free
        loss = loss_rec + LAMBDA_DC * loss_dc

        opt.zero_grad(); loss.backward(); opt.step()
        hist.append((loss_rec.item(), loss_dc.item()))
        if it % LOG_EVERY == 0:
            print(f"  iter {it:5d} | rec {loss_rec.item():.5f} | dc {loss_dc.item():.4f} | A {A.flatten().detach().cpu().numpy().round(3)}")

    # final estimate (no input perturbation), then recover J at full resolution with the haze model
    with torch.no_grad():
        J_net.eval(); t_net.eval()
        t = T_MIN + (1 - T_MIN) * torch.sigmoid(t_net(z_t))
        A = torch.sigmoid(a_logit).view(1, 3, 1, 1)
        I_full = to_tensor(hazy)
        t_full = F.interpolate(t, size=I_full.shape[-2:], mode='bilinear', align_corners=False)
        J_full = ((I_full - A) / t_full + A).clamp(0, 1)

    return (J_full[0].permute(1, 2, 0).cpu().numpy(), t_full[0, 0].cpu().numpy(),
            A.flatten().cpu().numpy(), np.array(hist))


def dehaze(hazy):
    """hazy: PIL RGB image -> uint8 RGB array."""
    J, t, A, hist = dehaze_dip(hazy)
    return (J * 255 + 0.5).astype(np.uint8)

## Quick test (50 iterations)

In [ ]:
NUM_ITER, LOG_EVERY = 50, 10
J, t, A, hist = dehaze_dip(Image.open(hazy_paths[0]).convert('RGB'))
print(J.shape, t.shape, A, hist[-1])
NUM_ITER, LOG_EVERY = 2000, 500

## Run over the dataset

In [ ]:
import time
for p in hazy_paths:
    hazy = Image.open(p).convert('RGB')
    t0 = time.time()
    pred = dehaze(hazy)
    dt = time.time() - t0
    Image.fromarray(pred).save(f"{OUT}/{os.path.basename(p)}")
    print(f"{os.path.basename(p)} | {hazy.size[0]}x{hazy.size[1]} | {dt:.1f}s")

## Preview

In [ ]:
for p in hazy_paths[:1]:
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    for a, im, t in zip(ax, [Image.open(p), Image.open(f"{OUT}/{os.path.basename(p)}")], ['hazy', 'DIP']):
        a.imshow(im); a.set_title(t); a.axis('off')
    plt.tight_layout(); plt.show()

## Evaluate
Score the outputs with `dehazing/evaluation` (see `dehaze_eval_colab.ipynb`):
```python
per_image, summary = evaluate(DATA_ROOT, {"dip": OUT})
```